# Pregunta 3 — Entrenamiento YOLO en CCT-20 (Colab / Kaggle)

Flujo: entorno → código → datos → exploración → conversión → E1/E2 (vacías) → E3 (modelo principal) → evaluación cis/trans.

**Antes de correr**
- **Colab:** Entorno de ejecución → Cambiar tipo → GPU (T4). Los resultados se guardan en Google Drive (`MyDrive/pregunta3_runs`) para no perderlos si se corta la sesión.
- **Kaggle:** Settings → Accelerator: GPU T4 / P100, y **Internet: On**. Los resultados quedan en `/kaggle/working/runs`.
- Si la sesión se corta a mitad de entrenamiento, vuelve a correr todo: la descarga se salta lo que ya existe y el entrenamiento se reanuda desde `last.pt`.

In [1]:
# 1. Entorno y rutas
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
EN_KAGGLE = Path("/kaggle").exists()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE = Path("/content/pregunta3")
    DATA = Path("/content/data")                      # disco local: rapido
    RUNS = Path("/content/drive/MyDrive/pregunta3_runs")
elif EN_KAGGLE:
    CODE = Path("/kaggle/working/pregunta3")
    DATA = Path("/tmp/cct")                           # /kaggle/working solo tiene ~20 GB
    RUNS = Path("/kaggle/working/runs")
else:
    CODE = Path.cwd()
    DATA = CODE / "data"
    RUNS = CODE / "runs"

for p in (DATA, RUNS):
    p.mkdir(parents=True, exist_ok=True)
WORKERS = 2 if EN_COLAB else 4
print("colab:", EN_COLAB, "| kaggle:", EN_KAGGLE)
print("codigo:", CODE, "\ndatos:", DATA, "\nresultados:", RUNS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
colab: True | kaggle: True
codigo: /content/pregunta3 
datos: /content/data 
resultados: /content/drive/MyDrive/pregunta3_runs
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Código del proyecto
Opción A: pon la URL de tu repositorio de GitHub en `REPO_URL`.
Opción B: deja `REPO_URL` vacío y sube `pregunta3_codigo.zip` (en Colab: panel de archivos → `/content`; en Kaggle: como Dataset, queda en `/kaggle/input/...`).

In [3]:
REPO_URL = "https://github.com/roshan0297/rnc-pregunta3-parcial.git"
SUBCARPETA = ""   # carpeta dentro del repo donde esta src/ ("" = raiz)

if not (CODE / "src").exists():
    if REPO_URL:
        tmp = Path("/tmp/repo")
        if not tmp.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(tmp)], check=True)
        src_repo = tmp / SUBCARPETA if (tmp / SUBCARPETA).exists() else tmp
        subprocess.run(["cp", "-r", str(src_repo), str(CODE)], check=True)
    else:
        zips = list(Path("/content").glob("pregunta3_codigo.zip")) + list(Path("/kaggle/input").rglob("pregunta3_codigo.zip"))
        assert zips, "Sube pregunta3_codigo.zip o pon REPO_URL"
        subprocess.run(["unzip", "-oq", str(zips[0]), "-d", str(CODE)], check=True)

os.chdir(CODE)
sys.path.insert(0, str(CODE / "src"))
!ls {CODE} {CODE}/src

/content/pregunta3:
notebooks  README.md  requirements.txt	results  src

/content/pregunta3/src:
cct.py	convertir_yolo.py  descargar_datos.py  explorar.py


In [4]:
# 3. Dependencias
!pip install -q -r requirements.txt
import ultralytics
ultralytics.checks()

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.6/112.6 GB disk)


## 4. Descarga de datos
Imágenes CCT-20 (~6 GB) + 3000 vacías de ubicaciones que no están en CCT-20 (el split `train` no trae vacías). En Colab tarda ~10–20 min.

In [5]:
RAW = DATA / "raw"
!python src/descargar_datos.py --raw {RAW} --vacias-extra 3000 --borrar-tar
!du -sh {RAW}/*

  eccv_18_annotations.tar.gz: 0.00 / 0.00 GB
/content/pregunta3/src/descargar_datos.py:138: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(tar_anot).extractall(raw)
  eccv_18_all_images_sm.tar.gz: 6.49 / 6.49 GB
  descomprimiendo imagenes (tarda varios minutos)...
/content/pregunta3/src/descargar_datos.py:144: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(tar_imgs).extractall(raw / "imagenes")
  caltech_camera_traps.json.zip: 0.01 / 0.01 GB
  vacias extra: 3000/3000
  3000 vacias de 106 ubicaciones
8.6M	/content/data/raw/caltech_camera_traps.json.zip
26M	/content/data/raw/eccv_18_annotation_files
2.9M	/content/data/raw/eccv_18_annotations.tar.gz
6.3G	/content/data/raw/imagenes
0	/content/data/raw/imagenes_ok

In [6]:
# 5. Exploracion con imagenes (porcentaje nocturno y mosaico de ejemplos)
!python src/explorar.py --anot {RAW}/eccv_18_annotation_files --imgs {RAW}/imagenes --out {RUNS}/exploracion


Imagenes por split y estado:
 estado      con_caja  sin_caja  vacia  total
split                                       
train          12099      1454      0  13553
cis_val         1665       824    995   3484
trans_val       1533       131     61   1725
cis_test       12696      1951   1180  15827
trans_test     18033      3464   1778  23275

Cajas por especie (train / cis_test / trans_test):
 especie  train_cajas  cis_test_cajas  trans_test_cajas
 opossum         2514            4010              4663
  rabbit         2278            1559               722
  coyote         1371            1182              1902
     cat         1170            1528              1366
squirrel         1037             510               782
 raccoon         1030             981              4803
     dog          769             725               961
  bobcat          684             772              1979
     car          668             791              1113
    bird          560             752     

## 6. Conversión a YOLO
Dos versiones del dataset para el experimento de vacías: sin vacías y con 20 % de vacías extra. Val (`cis_val` + `trans_val`) y test son iguales en ambas. Las imágenes se enlazan (hardlink), no se copian.

In [7]:
YOLO_SIN = DATA / "yolo_sin_vacias"
YOLO_V20 = DATA / "yolo_vacias20"
COMUNES = f"--anot {RAW}/eccv_18_annotation_files --imgs {RAW}/imagenes --vacias-extra {RAW}/vacias_extra"

!python src/convertir_yolo.py {COMUNES} --salida {YOLO_SIN} --frac-vacias 0
!python src/convertir_yolo.py {COMUNES} --salida {YOLO_V20} --frac-vacias 0.2

# guardar los metadatos de la conversion junto a los resultados
!mkdir -p {RUNS}/conversion && cp {YOLO_V20}/metadatos.csv {YOLO_V20}/config_conversion.json {RUNS}/conversion/

clases: ['badger', 'bird', 'bobcat', 'cat', 'coyote', 'deer', 'dog', 'fox', 'opossum', 'rabbit', 'raccoon', 'rodent', 'skunk', 'squirrel']
excluidas: ['car']
train       11431 imagenes
cis_val      2619 imagenes
trans_val    1594 imagenes
cis_test    13085 imagenes
trans_test  18698 imagenes

imagenes por split y motivo ('' = incluida):
 motivo             animal_sin_caja  clase_excluida
split                                             
cis_test    13085             1951             791
cis_val      2619              824              41
train       11431             1454             668
trans_test  18698             3464            1113
trans_val    1594              131               0
clases: ['badger', 'bird', 'bobcat', 'cat', 'coyote', 'deer', 'dog', 'fox', 'opossum', 'rabbit', 'raccoon', 'rodent', 'skunk', 'squirrel']
excluidas: ['car']
train       11431 imagenes
cis_val      2619 imagenes
trans_val    1594 imagenes
cis_test    13085 imagenes
trans_test  18698 imagenes
train     

## 7. Entrenamiento
Hiperparámetros comunes. Cada decisión debe quedar justificada en el informe:
- `flipud=0`: los animales no aparecen de cabeza.
- `hsv_s` bajo (0.4 en vez de 0.7): buena parte de las fotos son infrarrojas (grises) y saturar color no aporta.
- `patience=10`: corta si el mAP de validación no mejora en 10 épocas.

In [13]:
import pandas as pd
from ultralytics import YOLO

COMUN = dict(
    imgsz=640, epochs=30, batch=32, workers=WORKERS, device=0, seed=42,
    patience=10, project=str(RUNS), exist_ok=True,
    fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    mosaic=1.0, close_mosaic=5,
)

def entrenar(nombre, yaml, modelo="yolov8n.pt", **cambios):
    # Entrena o reanuda. Devuelve la ruta de best.pt
    pesos = RUNS / nombre / "weights"
    if (pesos / "last.pt").exists():
        try:
            YOLO(pesos / "last.pt").train(resume=True)
        except AssertionError as e:   # ya habia terminado: no hay nada que reanudar
            print("ya entrenado:", e)
    else:
        YOLO(modelo).train(data=str(yaml), name=nombre, **{**COMUN, **cambios})
    return pesos / "best.pt"

def tasa_falsos_disparos(pesos, dir_yolo, splits=("cis_val", "trans_val"), conf=0.5, imgsz=640):
    # % de imagenes vacias donde el modelo detecta algo con confianza >= conf
    modelo = YOLO(pesos)
    filas = []
    for s in splits:
        vacias = [dir_yolo / "images" / s / (t.stem + ".jpg")
                  for t in (dir_yolo / "labels" / s).glob("*.txt") if t.stat().st_size == 0]
        disparos = 0
        # en lotes de 32: una lista grande de rutas se procesa como un solo lote y llena la GPU
        for i in range(0, len(vacias), 32):
            res = modelo.predict(vacias[i:i + 32], conf=conf, imgsz=imgsz, verbose=False)
            disparos += sum(len(r.boxes) > 0 for r in res)
        filas.append({"split": s, "n_vacias": len(vacias), "con_disparo": disparos,
                      "tasa": disparos / max(len(vacias), 1)})
    return pd.DataFrame(filas)

### E1 y E2: ¿agregar vacías ayuda?
yolov8n (rápido, ~1 h cada uno en T4). Misma configuración; lo único que cambia son las vacías en train.

In [9]:
best_e1 = entrenar("E1_n640_sin_vacias", YOLO_SIN / "cct.yaml")

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/yolo_sin_vacias/cct.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E1_n640_sin_vacias, nbs=64, nms=None, opset=None, op

In [10]:
best_e2 = entrenar("E2_n640_vacias20", YOLO_V20 / "cct.yaml")

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/yolo_vacias20/cct.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E2_n640_vacias20, nbs=64, nms=None, opset=None, optimi

In [12]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

In [14]:
# Comparacion E1 vs E2 sobre validacion: mAP y falsos disparos en vacias
filas = []
for nombre, best, d in [("E1", best_e1, YOLO_SIN), ("E2", best_e2, YOLO_V20)]:
    m = YOLO(best).val(data=str(d / "cct.yaml"), split="val", imgsz=640, verbose=False, plots=False)
    fd = tasa_falsos_disparos(best, d)
    filas.append({"exp": nombre, "mAP50": m.box.map50, "mAP50-95": m.box.map,
                  **{f"falsos_{r.split}": r.tasa for r in fd.itertuples()}})
comp = pd.DataFrame(filas)
comp.to_csv(RUNS / "comparacion_E1_E2.csv", index=False)
comp

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,135,866 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 493.3±909.9 MB/s, size: 106.3 KB)
val: Scanning /content/data/yolo_sin_vacias/labels/cis_val.cache... 4213 images, 1056 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4213/4213 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 264/264 6.5it/s 40.6s
                   all       4213       3395      0.772       0.65       0.73      0.486
Speed: 0.8ms preprocess, 3.1ms inference, 0.0ms loss, 1.3ms postprocess per image
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,135,866 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1556.4±1270.5 MB/s, size: 119.2 KB)
val: Scanning /content/data/yolo_vacias20/labels/

,exp,mAP50,mAP50-95,falsos_cis_val,falsos_trans_val
0,E1,0.729707,0.485964,0.060302,0.081967
1,E2,0.747859,0.514085,0.034171,0.049180


### E3: modelo principal
Elige la configuración según E1/E2 (y el histograma de tamaños de la exploración) y anota el porqué en la bitácora.
`yolov8s` a 640 con 50 épocas tarda ~4–6 h en T4: lánzalo y deja la pestaña abierta (Colab) o usa "Save & Run All" (Kaggle).

In [ ]:
DATASET_E3 = YOLO_V20      # o YOLO_SIN, segun E1 vs E2
MODELO_E3 = "yolov8s.pt"
IMGSZ_E3 = 640             # 960 si las cajas pequenas pesan mucho y la GPU lo permite (batch 8)

best_e3 = entrenar("E3_principal", DATASET_E3 / "cct.yaml", MODELO_E3,
                   imgsz=IMGSZ_E3, epochs=50, batch=16 if IMGSZ_E3 == 640 else 8)

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/yolo_vacias20/cct.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E3_principal, nbs=64, nms=None, opset=None, optimize=F

## 8. Evaluación final: cis_test vs trans_test
mAP@0.5 y mAP@0.5:0.95 global y por especie, más la tasa de falsos disparos en las vacías de test. Esto alimenta los entregables 2 y 3.

In [ ]:
def evaluar(best, dir_yolo, imgsz, nombre_exp):
    tablas = []
    for yaml, test in [("cct.yaml", "cis_test"), ("cct_trans.yaml", "trans_test")]:
        m = YOLO(best).val(data=str(dir_yolo / yaml), split="test", imgsz=imgsz,
                           project=str(RUNS / nombre_exp), name=f"eval_{test}", exist_ok=True)
        t = pd.DataFrame({
            "especie": [m.names[i] for i in m.box.ap_class_index],
            "AP50": m.box.ap50, "AP50-95": m.box.ap,
        })
        t.loc[len(t)] = ["TODAS (mAP)", m.box.map50, m.box.map]
        t.insert(0, "test", test)
        tablas.append(t)
    res = pd.concat(tablas)
    res.to_csv(RUNS / nombre_exp / "eval_por_especie.csv", index=False)
    fd = tasa_falsos_disparos(best, dir_yolo, splits=("cis_test", "trans_test"), imgsz=imgsz)
    fd.to_csv(RUNS / nombre_exp / "falsos_disparos_test.csv", index=False)
    return res, fd

res, fd = evaluar(best_e3, DATASET_E3, IMGSZ_E3, "E3_principal")
display(res.pivot(index="especie", columns="test", values="AP50").round(3))
display(fd)